# 03. 3D TransUNet Training

This notebook trains a 3D TransUNet from `.pt` tensors.

## Supported `.pt` shapes
- Per-sample: `[D, H, W]` or `[1, D, H, W]`
- Batched/sharded: `[N, D, H, W]` or `[N, 1, D, H, W]`

`X` should be `float32`.

`Y` should be integer class IDs.
- `int8` is only safe up to 127 classes.
- For ~200 classes, store labels as `uint8` or `int16`.

In [1]:
import os
import random
import time
from pathlib import Path

import numpy as np

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
from tqdm.auto import tqdm

from scalesurfer.config import (
    DATA_CFG,
    DATA_PATH,
    DEVICE,
    MODEL_CFG,
    PATCH_SIZE,
    SEED,
    TRAIN_CFG,
    build_runtime_cfgs,
)
from scalesurfer.utils import gpu_mem_gb
from scalesurfer.splits import (
    DEFAULT_NOTEBOOK_SPLIT_MANIFEST_PATH,
    build_or_load_study_split_manifest,
    discover_training_pairs,
    extract_study_id,
    split_from_manifest,
)
from scalesurfer.api import prepare_data_pipeline, summarize_data_pipeline
from scalesurfer.volume.model import TransUNet3D
from scalesurfer.train import evaluate_runtime, init_training_runtime, step_scheduler_epoch, train_step_runtime

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # torch.backends.cudnn.benchmark = False
    # torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
# torch.use_deterministic_algorithms(True, warn_only=True)

In [2]:
file_map = discover_training_pairs(
    DATA_PATH / "tensors",
    x_filename="orig.pt",
    y_filename="aparc+aseg.pt",
)
study_counts = {}
for paths in file_map.values():
    study = extract_study_id(paths["orig"])
    study_counts[study] = study_counts.get(study, 0) + 1

SPLIT_MANIFEST_PATH = DEFAULT_NOTEBOOK_SPLIT_MANIFEST_PATH
{
    "n_pairs": len(file_map),
    "n_studies": len(study_counts),
    "study_preview": sorted(study_counts.items())[:10],
    "manifest_path": str(SPLIT_MANIFEST_PATH),
}


{'n_pairs': 4241,
 'n_studies': 53,
 'study_preview': [('ds001734', 117),
  ('ds002041', 25),
  ('ds002547', 26),
  ('ds002785', 216),
  ('ds002790', 226),
  ('ds002799', 26),
  ('ds003097', 928),
  ('ds003354', 8),
  ('ds003495', 24),
  ('ds003633', 11)],
 'manifest_path': '/home/rph/scalesurfer/docs/notebooks/01_volume/04_train_split_manifest.json'}

In [3]:
split_manifest = build_or_load_study_split_manifest(
    data_root=DATA_PATH,
    manifest_path=SPLIT_MANIFEST_PATH,
    x_filename="orig.pt",
    y_filename="aparc+aseg.pt",
    seed=SEED,
    ratios=(0.8, 0.1, 0.1),
    force_rebuild=False,
)
split = split_from_manifest(split_manifest, split_key="train_notebook_split")

DATA_OVERRIDES = {
    "x_train_files": split["x_train"],
    "y_train_files": split["y_train"],
    "x_val_files": split["x_val"],
    "y_val_files": split["y_val"],
    "x_test_files": split["x_test"],
    "y_test_files": split["y_test"],
    "group_split_enabled": False,
    "split_seed": SEED,
}
MODEL_OVERRIDES = {
    # Smaller/faster backbone while keeping transformer context.
    "channels": (12, 20, 32, 48, 64, 96),
    "transformer_depth": 2,
    "n_heads": 4,
    "dropout": 0.0,
    "positional_encoding": "sincos",
}
TRAIN_OVERRIDES = {
    # Rapid training profile.
    "epochs": 25,
    "effective_batch_size": 1,
    "initial_micro_batch_size": 1,
    "patch_chunk_size": 192,
    "quick_val_every_steps": 500,
    "quick_val_batches": 1,
    "early_stopping_patience": 6,
    "lr": 1e-3,
    "lr_scheduler": "cosine_warmup",
    "warmup_steps": 0,
    "cosine_total_steps": None,
    "weight_decay": 5e-5,
    "target_max_vram_gb": 24.0,
    "num_workers": 8,
    "prefetch_factor": 2,
    # Full-volume pass so transformer attention is global across all tokens.
    "spatial_window": None,
    "spatial_stride": None,
    "enforce_global_attention": True,
}

DATA_CFG, MODEL_CFG, TRAIN_CFG, CFG_CHANGED = build_runtime_cfgs(
    data_overrides=DATA_OVERRIDES,
    model_overrides=MODEL_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
)

print(f"device={DEVICE}")
print(
    f"split sizes -> train={len(split['x_train'])} "
    f"val={len(split['x_val'])} test={len(split['x_test'])}"
)
print(f"split manifest: {split_manifest['manifest_path']}")
print(f"split seed: {split_manifest['split_strategy']['seed']}")
print("cfg overrides:", {k: sorted(v.keys()) for k, v in CFG_CHANGED.items()})


device=cuda
split sizes -> train=3389 val=426 test=426
split manifest: /home/rph/scalesurfer/docs/notebooks/01_volume/04_train_split_manifest.json
split seed: 1337
cfg overrides: {'data': ['x_test_files', 'x_train_files', 'x_val_files', 'y_test_files', 'y_train_files', 'y_val_files'], 'model': ['channels', 'dropout', 'positional_encoding'], 'train': ['early_stopping_patience', 'effective_batch_size', 'enforce_global_attention', 'epochs', 'lr', 'lr_scheduler', 'num_workers', 'patch_chunk_size', 'quick_val_batches', 'quick_val_every_steps', 'spatial_window', 'target_max_vram_gb', 'weight_decay']}


In [4]:
pipeline = prepare_data_pipeline(DATA_CFG, TRAIN_CFG)

steps_per_epoch = len(pipeline.train_loader)
if TRAIN_CFG.get("lr_scheduler", "plateau") == "cosine_warmup":
    if int(TRAIN_CFG.get("warmup_steps", 0)) <= 0:
        # One full epoch warmup for large datasets.
        TRAIN_CFG["warmup_steps"] = int(steps_per_epoch)
    if TRAIN_CFG.get("cosine_total_steps") is None:
        TRAIN_CFG["cosine_total_steps"] = int(TRAIN_CFG["epochs"]) * int(steps_per_epoch)
    print(
        f"cosine schedule -> steps/epoch={steps_per_epoch}, "
        f"warmup_steps={TRAIN_CFG['warmup_steps']}, "
        f"total_steps={TRAIN_CFG['cosine_total_steps']}"
    )

class_values = pipeline.class_values
label_lut = pipeline.label_lut
label_values = class_values.tolist()

summarize_data_pipeline(pipeline)

cosine schedule -> steps/epoch=3388, warmup_steps=3388, total_steps=84700
train samples: 3388
val samples:   426
test samples:  426
volume shape (padded): (256, 256, 256)
inferred classes: 118
n_classes used:  118
label source:    default aparc+aseg labels
cache enabled:   True
cache dir:       /home/rph/scalesurfer/.tensor_cache_preproc
cache x dtype:   float16
cache y dtype:   int16
cache built now: 0
cache reused:    4240
cache skipped:   1
effective batch size: 1
max_open_files: 1


In [5]:
# Training loop: adaptive micro-batch, adaptive LR, frequent val, best checkpoint
model = TransUNet3D(
    n_classes=pipeline.n_classes,
    in_channels=int(MODEL_CFG["in_channels"]),
    base_shape=tuple(int(v) for v in pipeline.base_volume_shape),
    patch_size=PATCH_SIZE,
    channels=tuple(int(v) for v in MODEL_CFG["channels"]),
    transformer_depth=int(MODEL_CFG["transformer_depth"]),
    n_heads=int(MODEL_CFG["n_heads"]),
    dropout=float(MODEL_CFG["dropout"]),
    positional_encoding=str(MODEL_CFG["positional_encoding"]),
).to(DEVICE)

model = torch.compile(model)

runtime = init_training_runtime(model, TRAIN_CFG)

quick_val_every_steps = int(TRAIN_CFG["quick_val_every_steps"])
quick_val_batches = int(TRAIN_CFG["quick_val_batches"])
target_vram = TRAIN_CFG.get("target_max_vram_gb", None)
auto_increase_micro_batch = bool(TRAIN_CFG["auto_increase_micro_batch"])
patience = int(TRAIN_CFG["early_stopping_patience"])
effective_batch_size = pipeline.effective_batch_size
ckpt_path = runtime.ckpt_path

epochs = int(TRAIN_CFG["epochs"])
best_val = float("inf")
best_epoch = 0
epochs_no_improve = 0
global_step = 0
history = []
for epoch in range(1, epochs + 1):
    epoch_start = time.time()
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    model.train()
    train_num, train_den = 0.0, 0
    epoch_hit_oom = False

    train_pbar = tqdm(pipeline.train_loader, total=len(pipeline.train_loader), desc=f"epoch {epoch:03d} train", leave=False)
    for x_cpu, y_cpu in train_pbar:
        batch_num, batch_den, hit_oom = train_step_runtime(model, runtime, x_cpu, y_cpu)
        epoch_hit_oom = epoch_hit_oom or hit_oom

        train_num += batch_num
        train_den += batch_den
        global_step += 1

        if (
            quick_val_every_steps > 0
            and pipeline.val_loader is not None
            and global_step % quick_val_every_steps == 0
        ):
            quick_val = evaluate_runtime(
                model,
                pipeline.val_loader,
                runtime,
                max_batches=quick_val_batches,
                desc=f"quick val@{global_step}",
            )
            print(
                f"step {global_step:06d} | quick_val_ce: {quick_val:.4f} | "
                f"lr: {runtime.optimizer.param_groups[0]['lr']:.2e} | "
                f"micro_bs: {runtime.runtime_micro_bs} | patch_chunk: {runtime.runtime_patch_chunk}"
            )

        alloc_gb, peak_gb = gpu_mem_gb()
        if DEVICE == "cuda" and target_vram is not None and alloc_gb > float(target_vram):
            if runtime.runtime_micro_bs > 1:
                runtime.runtime_micro_bs = max(1, runtime.runtime_micro_bs // 2)
                print(
                    f"[VRAM] alloc {alloc_gb:.1f}GB > target {target_vram}GB -> "
                    f"micro_batch_size {runtime.runtime_micro_bs}"
                )
            elif runtime.runtime_patch_chunk > 16:
                runtime.runtime_patch_chunk = max(16, runtime.runtime_patch_chunk // 2)
                print(
                    f"[VRAM] alloc {alloc_gb:.1f}GB > target {target_vram}GB -> "
                    f"patch_chunk_size {runtime.runtime_patch_chunk}"
                )

        train_pbar.set_postfix(
            loss=f"{(train_num / max(1, train_den)):.4f}",
            lr=f"{runtime.optimizer.param_groups[0]['lr']:.2e}",
            mb=runtime.runtime_micro_bs,
            pc=runtime.runtime_patch_chunk,
            vram=f"{alloc_gb:.1f}/{peak_gb:.1f}G",
        )

    train_pbar.close()

    train_loss = train_num / max(1, train_den)
    if pipeline.val_loader is not None:
        val_loss = evaluate_runtime(
            model,
            pipeline.val_loader,
            runtime,
            desc=f"epoch {epoch:03d} val",
        )
    else:
        val_loss = train_loss
    step_scheduler_epoch(runtime, val_loss)

    epoch_sec = time.time() - epoch_start
    alloc_gb, peak_gb = gpu_mem_gb()

    history.append(
        {
            "epoch": epoch,
            "train_ce": float(train_loss),
            "val_ce": float(val_loss),
            "lr": float(runtime.optimizer.param_groups[0]["lr"]),
            "micro_bs": int(runtime.runtime_micro_bs),
            "patch_chunk": int(runtime.runtime_patch_chunk),
            "window": tuple(runtime.runtime_window) if runtime.runtime_window is not None else None,
            "sec": float(epoch_sec),
        }
    )

    print(
        f"epoch {epoch:03d} | train_ce: {train_loss:.4f} | val_ce: {val_loss:.4f} | "
        f"lr: {runtime.optimizer.param_groups[0]['lr']:.2e} | t: {epoch_sec:.1f}s | "
        f"micro_bs: {runtime.runtime_micro_bs} | patch_chunk: {runtime.runtime_patch_chunk} | "
        f"window: {runtime.runtime_window} | vram: {alloc_gb:.1f}/{peak_gb:.1f}GB"
    )

    if val_loss < best_val:
        best_val = float(val_loss)
        best_epoch = int(epoch)
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": best_epoch,
                "best_val": best_val,
                "model_state": model.state_dict(),
                "optimizer_state": runtime.optimizer.state_dict(),
                "scheduler_state": runtime.scheduler.state_dict(),
                "scaler_state": runtime.scaler.state_dict() if runtime.scaler.is_enabled() else None,
                "model_cfg": MODEL_CFG,
                "train_cfg": TRAIN_CFG,
                "data_cfg": DATA_CFG,
                "patch_size": PATCH_SIZE,
                "base_volume_shape": pipeline.base_volume_shape,
                "n_classes": pipeline.n_classes,
                "runtime_micro_bs": runtime.runtime_micro_bs,
                "runtime_patch_chunk": runtime.runtime_patch_chunk,
                "runtime_window": runtime.runtime_window,
                "history": history,
            },
            ckpt_path,
        )
        print(f"saved best checkpoint -> {ckpt_path} (epoch={best_epoch}, val_ce={best_val:.4f})")
    else:
        epochs_no_improve += 1

    if auto_increase_micro_batch and (not epoch_hit_oom) and runtime.runtime_micro_bs < effective_batch_size:
        runtime.runtime_micro_bs = min(effective_batch_size, runtime.runtime_micro_bs * 2)
        print(f"no OOM in epoch {epoch:03d}; micro_batch_size increased -> {runtime.runtime_micro_bs}")

    if patience > 0 and epochs_no_improve >= patience:
        print(f"early stopping at epoch {epoch:03d} (no val improvement for {patience} epochs)")
        break

print(f"best epoch: {best_epoch}, best val_ce: {best_val:.4f}, checkpoint: {ckpt_path}")

# Final test evaluation using best checkpoint
if ckpt_path.exists():
    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state"])

if pipeline.test_loader is not None and len(pipeline.test_loader) > 0:
    test_loss = evaluate_runtime(model, pipeline.test_loader, runtime, desc="final test")
    print(f"final test_ce (best ckpt): {test_loss:.4f}")
else:
    print("final test skipped (no test samples)")


epoch 001 train:   0%|          | 0/3388 [00:00<?, ?it/s]

quick val@500:   0%|          | 0/1 [00:00<?, ?it/s]

step 000500 | quick_val_ce: 1.6509 | lr: 1.49e-04 | micro_bs: 1 | patch_chunk: 192


quick val@1000:   0%|          | 0/1 [00:00<?, ?it/s]

step 001000 | quick_val_ce: 0.2816 | lr: 2.96e-04 | micro_bs: 1 | patch_chunk: 192


...

step 084500 | quick_val_ce: 0.0203 | lr: 1.01e-06 | micro_bs: 1 | patch_chunk: 192


epoch 025 val:   0%|          | 0/426 [00:00<?, ?it/s]

epoch 025 | train_ce: 0.0287 | val_ce: 0.0307 | lr: 1.00e-06 | t: 2922.8s | micro_bs: 1 | patch_chunk: 192 | window: None | vram: 0.1/25.1GB
best epoch: 22, best val_ce: 0.0300, checkpoint: checkpoints/transunet3d_best.pt


final test:   0%|          | 0/426 [00:00<?, ?it/s]

final test_ce (best ckpt): 0.0311


In [ ]:
# how to load trained model:
# ckpt = torch.load(TRAIN_CFG["checkpoint_path"], map_location="cpu")
# model.load_state_dict(ckpt["model_state"])
# print(f"Loaded epoch={ckpt['epoch']} best_val={ckpt['best_val']:.4f}")